In [60]:

import pandas as pd
import numpy as np
from sklearn.cluster import dbscan
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

In [55]:
data = pd.read_excel('data/Таблицы [16-02-2026 17-25-06] Корпус А ПР-103.xlsx', header=4, index_col=0)
time_labels = [['2026-02-05 09:00:17', '2026-02-05 10:00:17'],
               ['2026-02-05 18:00:17', '2026-02-05 20:00:17'],
               ['2026-02-06 23:00:17', '2026-02-09 14:00:17'],
               ['2026-02-16 13:00:17', '2026-02-16 15:00:17']
]

In [4]:
data.head()

,ВоздухУст-1 (UID1073741936),ВоздухУст-2 (UID1073741937),ВоздухУст-3 (UID1073741938),Воздух-1 (UID1073741837),Воздух-2 (UID1073741858),Воздух-3 (UID1073741859),ПодЦель-1 (UID1073741882),ПодЦель-2 (UID1073741883),ПодЦель-3 (UID1073741884),Подача-1 (UID1073741836),Подача-2 (UID1073741856),Подача-3 (UID1073741857),Кран-1% (UID1073741835),Кран-2% (UID1073741854),Кран-3% (UID1073741855)
Дата/время,,,,,,,,,,,,,,,
2026-02-02 17:25:17,17.0,19.0,18.8,17.14,19.17,18.92,46.11,46.97,68.0,46.76,47.56,67.62,32.67,27.67,41.0
2026-02-02 17:25:47,17.0,19.0,18.8,17.15,19.17,18.92,46.11,46.97,68.0,46.84,47.55,67.38,32.67,27.67,41.0
2026-02-02 17:25:48,17.0,19.0,18.8,17.16,19.17,18.92,46.11,46.97,68.0,46.84,47.55,67.38,32.67,27.67,41.0
2026-02-02 17:26:18,17.0,19.0,18.8,17.07,19.18,18.92,46.11,46.97,68.0,46.98,47.68,67.28,32.67,27.67,41.0
2026-02-02 17:26:48,17.0,19.0,18.8,17.14,19.17,18.92,46.11,46.97,68.0,47.07,47.86,67.37,32.67,27.67,41.0


In [7]:
data.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 75818 entries, 2026-02-02 17:25:17 to 2026-02-16 17:24:50
Data columns (total 15 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ВоздухУст-1 (UID1073741936)  75810 non-null  float64
 1   ВоздухУст-2 (UID1073741937)  75812 non-null  float64
 2   ВоздухУст-3 (UID1073741938)  75812 non-null  float64
 3   Воздух-1 (UID1073741837)     75810 non-null  float64
 4   Воздух-2 (UID1073741858)     75810 non-null  float64
 5   Воздух-3 (UID1073741859)     75810 non-null  float64
 6   ПодЦель-1 (UID1073741882)    75811 non-null  float64
 7   ПодЦель-2 (UID1073741883)    75810 non-null  float64
 8   ПодЦель-3 (UID1073741884)    75810 non-null  float64
 9   Подача-1 (UID1073741836)     75811 non-null  float64
 10  Подача-2 (UID1073741856)     75810 non-null  float64
 11  Подача-3 (UID1073741857)     75810 non-null  float64
 12  Кран-1% (UID1073741835)      75814 non-

In [57]:
def create_labels(data, list_labels, use_horizon = False, horizon = None):
    df = data.copy()
    df['target'] = 0
    for i in list_labels:
        if use_horizon:
            df.loc[i[0]-horizon, 'target'] = 1
        else:
            df.loc[i[0]:i[1], 'target'] = 1
    return df

def preprocessing(data, list_labels, use_horizon=False, horizon = None):
    df = data.copy()
    df = df.dropna()
    #df.index = df.index.floor('min')
    #df = df[~df.index.duplicated(keep='first')]
    df = df.asfreq('30s')
    df = df.interpolate(method='linear')
    df = create_labels(df, list_labels, horizon=horizon, use_horizon=use_horizon)
    df, target = df.iloc[:, :-1], df.iloc[:, -1:]
    return df, target

X, y = preprocessing(data, list_labels=time_labels, use_horizon=False)

In [210]:
split_idx = int(len(X) * 0.8)
X_train = X.iloc[:split_idx].copy()
y_train = y.iloc[:split_idx].copy()
X_test = X.iloc[split_idx:].copy()
y_test = y.iloc[split_idx:].copy()

In [213]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 32256 entries, 2026-02-02 17:25:17 to 2026-02-13 22:12:47
Freq: 30s
Data columns (total 15 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ВоздухУст-1 (UID1073741936)  32256 non-null  float64
 1   ВоздухУст-2 (UID1073741937)  32256 non-null  float64
 2   ВоздухУст-3 (UID1073741938)  32256 non-null  float64
 3   Воздух-1 (UID1073741837)     32256 non-null  float64
 4   Воздух-2 (UID1073741858)     32256 non-null  float64
 5   Воздух-3 (UID1073741859)     32256 non-null  float64
 6   ПодЦель-1 (UID1073741882)    32256 non-null  float64
 7   ПодЦель-2 (UID1073741883)    32256 non-null  float64
 8   ПодЦель-3 (UID1073741884)    32256 non-null  float64
 9   Подача-1 (UID1073741836)     32256 non-null  float64
 10  Подача-2 (UID1073741856)     32256 non-null  float64
 11  Подача-3 (UID1073741857)     32256 non-null  float64
 12  Кран-1% (UID1073741835)      

In [236]:
import plotly.express as px
df = pd.concat([X, y], axis=1)
split_idx = int(len(df) * 0.7)
df_train = df.iloc[:split_idx].copy()
df_test = df.iloc[split_idx:].copy()
# Распределение классов в train
class_counts = df_train['target'].value_counts().reset_index()
class_counts.columns = ['класс', 'количество']
fig = px.bar(class_counts, x='класс', y='количество', title='Распределение классов в обучающей выборке')
fig.show()

print("Доля аномалий (класс 1) в train:", (df_train['target'] == 1).mean())

# Если есть тест с метками
if 'target' in df_test.columns:
    test_counts = df_test['target'].value_counts().reset_index()
    test_counts.columns = ['класс', 'количество']
    fig = px.bar(test_counts, x='класс', y='количество', title='Распределение классов в тестовой выборке')
    fig.show()
    print("Доля аномалий в test:", (df_test['target'] == 1).mean())

Доля аномалий (класс 1) в train: 0.280718537414966


Доля аномалий в test: 0.0199239417989418


In [ ]:
# Выберем несколько признаков (если их много, можно построить по одному или использовать subplots)
features = [col for col in df_train.columns]

# Гистограммы для train (можно добавить test для сравнения)
for feat in features[:5]:  # первые 5 для примера
    fig = go.Figure()
    fig.add_trace(go.Histogram(x=df_train[feat], name='train', opacity=0.7))
    if not df_test.empty:
        fig.add_trace(go.Histogram(x=df_test[feat], name='test', opacity=0.7))
    fig.update_layout(title=f'Распределение признака {feat}', barmode='overlay')
    fig.show()

In [ ]:
import plotly.graph_objects as go

for feat in features[:5]:
    fig = go.Figure()
    fig.add_trace(go.Box(y=df_train[feat], name='train'))
    if not df_test.empty:
        fig.add_trace(go.Box(y=df_test[feat], name='test'))
    fig.update_layout(title=f'Boxplot признака {feat}')
    fig.show()

In [ ]:
for feat in features[:5]:
    fig = go.Figure()
    fig.add_trace(go.Histogram(x=df_train[df_train['target']==0][feat], name='норма', opacity=0.7))
    fig.add_trace(go.Histogram(x=df_train[df_train['target']==1][feat], name='аномалия', opacity=0.7))
    fig.update_layout(title=f'Признак {feat} по классам (train)', barmode='overlay')
    fig.show()

In [ ]:
for feat in features[:5]:
    fig = go.Figure()
    fig.add_trace(go.Box(y=df_train[df_train['target']==0][feat], name='норма'))
    fig.add_trace(go.Box(y=df_train[df_train['target']==1][feat], name='аномалия'))
    fig.update_layout(title=f'Признак {feat} по классам (train)')
    fig.show()

In [230]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Масштабируем все данные вместе
scaler = StandardScaler()
X_combined = np.vstack([df_train[features], df_test[features]])
X_combined_scaled = scaler.fit_transform(X_combined)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_combined_scaled)

df_pca = pd.DataFrame({
    'PC1': X_pca[:, 0],
    'PC2': X_pca[:, 1],
    'source': ['train'] * len(df_train) + ['test'] * len(df_test)
})

fig = px.scatter(df_pca, x='PC1', y='PC2', color='source', title='PCA проекция train vs test')
fig.show()

In [237]:
from sklearn.manifold import TSNE

# Для ускорения возьмём случайные 5000 точек из train
np.random.seed(42)
sample_idx = np.random.choice(len(df_train), min(20000, len(df_train)), replace=False)
X_sample = df_train[features].iloc[sample_idx]
y_sample = df_train['target'].iloc[sample_idx]

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_sample)

df_tsne = pd.DataFrame({'tSNE1': X_tsne[:, 0], 'tSNE2': X_tsne[:, 1], 'target': y_sample})
fig = px.scatter(df_tsne, x='tSNE1', y='tSNE2', color='target',
                 title='t-SNE проекция обучающей выборки (подвыборка)')
fig.show()

In [238]:
df.columns

Index(['ВоздухУст-1 (UID1073741936)', 'ВоздухУст-2 (UID1073741937)',
       'ВоздухУст-3 (UID1073741938)', 'Воздух-1 (UID1073741837)',
       'Воздух-2 (UID1073741858)', 'Воздух-3 (UID1073741859)',
       'ПодЦель-1 (UID1073741882)', 'ПодЦель-2 (UID1073741883)',
       'ПодЦель-3 (UID1073741884)', 'Подача-1 (UID1073741836)',
       'Подача-2 (UID1073741856)', 'Подача-3 (UID1073741857)',
       'Кран-1% (UID1073741835)', 'Кран-2% (UID1073741854)',
       'Кран-3% (UID1073741855)', 'target'],
      dtype='object')

In [239]:
# Исходные колонки (без target)
feature_cols = [col for col in df_train.columns if col != 'target']

# Сгруппируем по типу (по окончанию имени после последнего пробела или по UID, но проще по смыслу)
# Из названий видно:
# ВоздухУст-1,2,3
# Воздух-1,2,3
# ПодЦель-1,2,3
# Подача-1,2,3
# Кран-1%,2%,3%

# Создадим словари для удобства
air_set_cols = ['ВоздухУст-1 (UID1073741936)', 'ВоздухУст-2 (UID1073741937)', 'ВоздухУст-3 (UID1073741938)']
air_actual_cols = ['Воздух-1 (UID1073741837)', 'Воздух-2 (UID1073741858)', 'Воздух-3 (UID1073741859)']
target_set_cols = ['ПодЦель-1 (UID1073741882)', 'ПодЦель-2 (UID1073741883)', 'ПодЦель-3 (UID1073741884)']
feed_actual_cols = ['Подача-1 (UID1073741836)', 'Подача-2 (UID1073741856)', 'Подача-3 (UID1073741857)']
valve_cols = ['Кран-1% (UID1073741835)', 'Кран-2% (UID1073741854)', 'Кран-3% (UID1073741855)']

# Функция для генерации признаков на одном датафрейме
def engineer_features(df):
    df_new = df.copy()
    
    # 1. Отклонения факта от уставки для воздуха
    for i in range(3):
        df_new[f'air_diff_{i+1}'] = df[air_actual_cols[i]] - df[air_set_cols[i]]
    
    # 2. Отклонения факта от цели для подачи
    for i in range(3):
        df_new[f'feed_diff_{i+1}'] = df[feed_actual_cols[i]] - df[target_set_cols[i]]
    
    # 3. Отношение подачи к крану (если кран не ноль)
    for i in range(3):
        df_new[f'feed_per_valve_{i+1}'] = df[feed_actual_cols[i]] / (df[valve_cols[i]] + 1e-8)  # защита от деления на 0
    
    # 4. Средние по линиям
    df_new['air_actual_mean'] = df[air_actual_cols].mean(axis=1)
    df_new['air_set_mean'] = df[air_set_cols].mean(axis=1)
    df_new['target_set_mean'] = df[target_set_cols].mean(axis=1)
    df_new['feed_actual_mean'] = df[feed_actual_cols].mean(axis=1)
    df_new['valve_mean'] = df[valve_cols].mean(axis=1)
    
    # 5. Стандартные отклонения по линиям (разброс)
    df_new['air_actual_std'] = df[air_actual_cols].std(axis=1)
    df_new['air_set_std'] = df[air_set_cols].std(axis=1)
    df_new['target_set_std'] = df[target_set_cols].std(axis=1)
    df_new['feed_actual_std'] = df[feed_actual_cols].std(axis=1)
    df_new['valve_std'] = df[valve_cols].std(axis=1)
    
    # 6. Разности между линиями (для примера, воздух факт: 1-2, 1-3, 2-3)
    df_new['air_diff_12'] = df[air_actual_cols[0]] - df[air_actual_cols[1]]
    df_new['air_diff_13'] = df[air_actual_cols[0]] - df[air_actual_cols[2]]
    df_new['air_diff_23'] = df[air_actual_cols[1]] - df[air_actual_cols[2]]
    
    df_new['feed_diff_12'] = df[feed_actual_cols[0]] - df[feed_actual_cols[1]]
    df_new['feed_diff_13'] = df[feed_actual_cols[0]] - df[feed_actual_cols[2]]
    df_new['feed_diff_23'] = df[feed_actual_cols[1]] - df[feed_actual_cols[2]]
    
    df_new['valve_diff_12'] = df[valve_cols[0]] - df[valve_cols[1]]
    df_new['valve_diff_13'] = df[valve_cols[0]] - df[valve_cols[2]]
    df_new['valve_diff_23'] = df[valve_cols[1]] - df[valve_cols[2]]
    
    # 7. Суммарные показатели
    df_new['air_actual_sum'] = df[air_actual_cols].sum(axis=1)
    df_new['feed_actual_sum'] = df[feed_actual_cols].sum(axis=1)
    
    # 8. Квадратичные признаки для наиболее коррелирующих (можно выбрать позже)
    # Например, квадраты отклонений
    for i in range(3):
        df_new[f'air_diff_{i+1}_squared'] = df_new[f'air_diff_{i+1}'] ** 2
        df_new[f'feed_diff_{i+1}_squared'] = df_new[f'feed_diff_{i+1}'] ** 2
    
    return df_new

# Применяем
df_train_eng = engineer_features(df_train)
df_test_eng = engineer_features(df_test)

# Теперь в df_train_eng и df_test_eng есть исходные признаки + новые
# Убедимся, что нет inf/NaN (деление на ноль мы защитили)
print("Новые признаки:", list(df_train_eng.columns[~df_train_eng.columns.isin(df_train.columns)]))

# После генерации новых признаков (код выше)
# Выделим признаки и целевую переменную
X_train_eng = df_train_eng.drop(columns=['target'])
y_train = df_train_eng['target']
X_test_eng = df_test_eng.drop(columns=['target'])
y_test = df_test_eng['target']  # если есть

# Масштабирование (для моделей, чувствительных к масштабу)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_eng)
X_test_scaled = scaler.transform(X_test_eng)

# Обучение модели (например, Random Forest с балансировкой)
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
model.fit(X_train_scaled, y_train)

# Оценка на тесте
from sklearn.metrics import classification_report
y_pred = model.predict(X_test_scaled)
print(classification_report(y_test, y_pred))

import plotly.express as px
importance = pd.DataFrame({
    'feature': X_train_eng.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
fig = px.bar(importance.head(20), x='importance', y='feature', orientation='h',
             title='Топ-20 важных признаков')
fig.show()

Новые признаки: ['air_diff_1', 'air_diff_2', 'air_diff_3', 'feed_diff_1', 'feed_diff_2', 'feed_diff_3', 'feed_per_valve_1', 'feed_per_valve_2', 'feed_per_valve_3', 'air_actual_mean', 'air_set_mean', 'target_set_mean', 'feed_actual_mean', 'valve_mean', 'air_actual_std', 'air_set_std', 'target_set_std', 'feed_actual_std', 'valve_std', 'air_diff_12', 'air_diff_13', 'air_diff_23', 'feed_diff_12', 'feed_diff_13', 'feed_diff_23', 'valve_diff_12', 'valve_diff_13', 'valve_diff_23', 'air_actual_sum', 'feed_actual_sum', 'air_diff_1_squared', 'feed_diff_1_squared', 'air_diff_2_squared', 'feed_diff_2_squared', 'air_diff_3_squared', 'feed_diff_3_squared']
              precision    recall  f1-score   support

           0       0.96      0.55      0.70     11855
           1       0.00      0.00      0.00       241

    accuracy                           0.54     12096
   macro avg       0.48      0.28      0.35     12096
weighted avg       0.95      0.54      0.69     12096



In [190]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report

# ======================
# 1. ПОДГОТОВКА ДАННЫХ
# ======================
# Предполагается, что переменные X_train, y_train, X_test, y_test уже определены
# Если y_test нет, можно оставить None

# Преобразуем в numpy массивы, если это pandas объекты
if hasattr(X_train, 'values'):
    X_train = X_train.values
if hasattr(y_train, 'values'):
    y_train = y_train.values
if hasattr(X_test, 'values'):
    X_test = X_test.values

# Масштабирование
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test) if X_test is not None else None

# ====================================
# 2. ПОДБОР ПАРАМЕТРОВ DBSCAN (опционально)
# ====================================
# Можно использовать k-distance график для выбора eps
# Возьмём min_samples = 2 * число признаков (эмпирически)
min_samples = 25  # можно регулировать

# Построим k-distance график
neigh = NearestNeighbors(n_neighbors=min_samples)
neigh.fit(X_train_scaled)
distances, _ = neigh.kneighbors(X_train_scaled)
k_dist = np.sort(distances[:, min_samples-1])

fig_elbow = go.Figure()
fig_elbow.add_trace(go.Scatter(x=np.arange(len(k_dist)), y=k_dist, mode='lines'))
fig_elbow.update_layout(title=f'k-distance график (k={min_samples})',
                        xaxis_title='Точки (отсортированы)',
                        yaxis_title=f'Расстояние до {min_samples}-го соседа')
fig_elbow.show()

# Пользователь должен выбрать eps в районе "локтя" графика.
# Для примера возьмём eps = 0.3 (можете подставить своё значение)
eps = 0.5  # измените на основе графика

# =============================
# 3. ОБУЧЕНИЕ DBSCAN НА TRAIN
# =============================
dbscan = DBSCAN(eps=eps, min_samples=min_samples)
train_labels = dbscan.fit_predict(X_train_scaled)

# Статистика по обучающей выборке
n_clusters = len(set(train_labels)) - (1 if -1 in train_labels else 0)
n_noise = list(train_labels).count(-1)
print(f"Обучающая выборка: всего точек {len(train_labels)}")
print(f"Найдено кластеров: {n_clusters}")
print(f"Аномалий (шума): {n_noise} ({n_noise/len(train_labels):.2%})")

# ==============================================
# 4. ПОДГОТОВКА МОДЕЛИ ДЛЯ ТЕСТОВЫХ ДАННЫХ
# ==============================================
# Используем все точки обучающей выборки, не являющиеся шумом,
# как эталон нормального поведения (ядровые + граничные)
normal_points = X_train_scaled[train_labels != -1]
if len(normal_points) == 0:
    raise ValueError("Нет ни одной нормальной точки в обучающей выборке! Увеличьте eps или уменьшите min_samples.")

# Обучаем NearestNeighbors с радиусом eps
nn = NearestNeighbors(n_neighbors=1, radius=eps).fit(normal_points)

def predict_anomaly(X_new_scaled):
    """Возвращает булев массив: True – аномалия, False – норма"""
    distances, _ = nn.radius_neighbors(X_new_scaled, radius=eps)
    return np.array([len(d) == 0 for d in distances])

# Применяем к тестовой выборке
if X_test_scaled is not None:
    test_anomaly_flags = predict_anomaly(X_test_scaled)
    print(f"\nТестовая выборка: всего точек {len(test_anomaly_flags)}")
    print(f"Предсказано аномалий: {np.sum(test_anomaly_flags)} ({np.mean(test_anomaly_flags):.2%})")

# =====================================
# 6. ОЦЕНКА КАЧЕСТВА (если есть метки)
# =====================================
if y_train is not None:
    print("\nОценка на обучающей выборке:")
    y_pred_train = (train_labels == -1).astype(int)
    print(classification_report(y_train, y_pred_train, target_names=['норма', 'аномалия']))

if y_test is not None and X_test_scaled is not None:
    print("\nОценка на тестовой выборке:")
    print(classification_report(y_test, test_anomaly_flags.astype(int), target_names=['норма', 'аномалия']))

Обучающая выборка: всего точек 32256
Найдено кластеров: 17
Аномалий (шума): 141 (0.44%)

Тестовая выборка: всего точек 8064
Предсказано аномалий: 7659 (94.98%)

Оценка на обучающей выборке:
              precision    recall  f1-score   support

       норма       0.76      1.00      0.86     24333
    аномалия       0.75      0.01      0.03      7923

    accuracy                           0.76     32256
   macro avg       0.75      0.51      0.44     32256
weighted avg       0.76      0.76      0.66     32256


Оценка на тестовой выборке:
              precision    recall  f1-score   support

       норма       1.00      0.05      0.10      7823
    аномалия       0.03      1.00      0.06       241

    accuracy                           0.08      8064
   macro avg       0.52      0.53      0.08      8064
weighted avg       0.97      0.08      0.10      8064



In [188]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

y_train = np.asarray(y_train).ravel()
train_anomaly = (train_labels == -1).ravel()
if len(train_anomaly) != len(y_train):
    raise ValueError("Длина train_labels не совпадает с y_train")

if y_test is not None:
    y_test = np.asarray(y_test).ravel()
    anomaly_flags = np.asarray(anomaly_flags).ravel()
    if len(anomaly_flags) != len(y_test):
        raise ValueError("Длина anomaly_flags не совпадает с y_test")

# Создаём фигуру с двумя подграфиками (если есть тест)
if y_test is not None:
    fig = make_subplots(rows=2, cols=1, 
                        subplot_titles=('Обучающая выборка', 'Тестовая выборка'),
                        shared_xaxes=False, vertical_spacing=0.15)
else:
    fig = make_subplots(rows=1, cols=1, subplot_titles=('Обучающая выборка'))

# Индексы для осей X
indices_train = np.arange(len(y_train))

# ---- Обучающая выборка ----
# Линия всех значений y_train
fig.add_trace(
    go.Scatter(x=indices_train, y=y_train, mode='lines',
               name='y_train', line=dict(color='blue', width=1)),
    row=1, col=1
)

# Точки, предсказанные как аномалии
anom_train_idx = indices_train[train_anomaly]
anom_train_vals = y_train[train_anomaly]
fig.add_trace(
    go.Scatter(x=anom_train_idx, y=anom_train_vals, mode='markers',
               name='Аномалии (предсказанные)',
               marker=dict(color='red', size=6, symbol='circle')),
    row=1, col=1
)

# Если y_train содержит бинарные метки аномалий (1 - аномалия), можно показать реальные
# Предположим, что y_train может быть не только вещественным, но и меткой
# Для примера: если y_train состоит из 0 и 1, то реальные аномалии - это y_train == 1
# Раскомментируйте, если нужно
# real_anom_train = (y_train == 1)
# if np.any(real_anom_train):
#     fig.add_trace(
#         go.Scatter(x=indices_train[real_anom_train], y=y_train[real_anom_train],
#                    mode='markers', name='Реальные аномалии (train)',
#                    marker=dict(symbol='circle-open', size=10, color='black', line=dict(width=2))),
#         row=1, col=1
#     )

# ---- Тестовая выборка (если есть) ----
if y_test is not None:
    indices_test = np.arange(len(y_test))
    
    fig.add_trace(
        go.Scatter(x=indices_test, y=y_test, mode='lines',
                   name='y_test', line=dict(color='green', width=1)),
        row=2, col=1
    )
    
    anom_test_idx = indices_test[anomaly_flags]
    anom_test_vals = y_test[anomaly_flags]
    fig.add_trace(
        go.Scatter(x=anom_test_idx, y=anom_test_vals, mode='markers',
                   name='Аномалии (предсказанные)',
                   marker=dict(color='red', size=6, symbol='circle')),
        row=2, col=1
    )
    
    # Реальные аномалии на тесте (если y_test бинарный)
    # real_anom_test = (y_test == 1)
    # if np.any(real_anom_test):
    #     fig.add_trace(
    #         go.Scatter(x=indices_test[real_anom_test], y=y_test[real_anom_test],
    #                    mode='markers', name='Реальные аномалии (test)',
    #                    marker=dict(symbol='circle-open', size=10, color='black', line=dict(width=2))),
    #         row=2, col=1
    #     )

# Настройка оформления
fig.update_layout(title='Обнаруженные аномалии на фоне значений y',
                  hovermode='x unified',
                  showlegend=True)

fig.update_xaxes(title_text='Индекс', row=1, col=1)
fig.update_yaxes(title_text='y_train', row=1, col=1)
if y_test is not None:
    fig.update_xaxes(title_text='Индекс', row=2, col=1)
    fig.update_yaxes(title_text='y_test', row=2, col=1)

fig.show()

In [168]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.decomposition import PCA

# Предполагаем, что у вас уже есть:
# X_train_scaled, X_test_scaled, train_labels, anomaly_flags

# Объединяем данные
X_test_scaled = scaler.transform(X_new)
X_combined = np.vstack([X_train_scaled, X_test_scaled])
source = ['train'] * len(X_train_scaled) + ['test'] * len(X_test_scaled)

# PCA для двух компонент
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_combined)

# Создаём DataFrame
df_pca = pd.DataFrame({
    'PC1': X_pca[:, 0],
    'PC2': X_pca[:, 1],
    'source': source,
    'anomaly_train': np.concatenate([(train_labels == -1), [np.nan] * len(X_test_scaled)]),
    'anomaly_test': np.concatenate([[np.nan] * len(X_train_scaled), anomaly_flags])
})

# Формируем категорию точки для окраски
df_pca['point_type'] = 'train_norm'
df_pca.loc[(df_pca['source'] == 'train') & (df_pca['anomaly_train'] == True), 'point_type'] = 'train_anomaly'
df_pca.loc[(df_pca['source'] == 'test') & (df_pca['anomaly_test'] == True), 'point_type'] = 'test_anomaly'
df_pca.loc[(df_pca['source'] == 'test') & (df_pca['anomaly_test'] == False), 'point_type'] = 'test_norm'

# Цветовая карта
color_map = {
    'train_norm': 'blue',
    'train_anomaly': 'red',
    'test_norm': 'green',
    'test_anomaly': 'orange'
}

# Интерактивный график
fig = px.scatter(
    df_pca, x='PC1', y='PC2', color='point_type',
    color_discrete_map=color_map,
    title='PCA проекция обучающих и тестовых данных',
    labels={'point_type': 'Тип точки'},
    hover_data={'source': True, 'anomaly_train': True, 'anomaly_test': True}
)
fig.show()

# Доля объяснённой дисперсии
print(f"Доля объяснённой дисперсии двумя компонентами: {pca.explained_variance_ratio_.sum():.3f}")

Доля объяснённой дисперсии двумя компонентами: 0.738


In [162]:
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# Предположим, X_train уже есть
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.values)

# Обучаем DBSCAN
dbscan = DBSCAN(eps=0.1, min_samples=17)
train_labels = dbscan.fit_predict(X_train_scaled)

# Получаем ядровые точки
core_samples_mask = np.zeros_like(train_labels, dtype=bool)
core_samples_mask[dbscan.core_sample_indices_] = True
core_points = X_train_scaled[core_samples_mask]

# Обучаем NearestNeighbors на ядровых точках
nn = NearestNeighbors(n_neighbors=1, radius=0.5).fit(core_points)

# Функция для классификации новых точек
def predict_new(X_new):
    X_new_scaled = scaler.transform(X_new)
    distances, _ = nn.radius_neighbors(X_new_scaled, radius=0.5)
    return np.array([len(d) == 0 for d in distances])  # True – аномалия

# Пример использования
X_new = X_test.values  # одна или несколько новых точек
anomaly_flags = predict_new(X_new)

In [194]:
from sklearn.metrics import f1_score
from sklearn.cluster import DBSCAN

best_f1 = 0
best_eps = None
best_min_samples = None

eps_range = np.linspace(0.1, 1.0, 20)
min_samples_range = range(3, 20)

for eps in eps_range:
    for min_samples in min_samples_range:
        dbscan = DBSCAN(eps=eps, min_samples=min_samples)
        pred_labels = dbscan.fit_predict(X_train_scaled)
        # Преобразуем: -1 -> аномалия (1), остальное -> норма (0)
        pred_anomaly = (pred_labels == 0).astype(int)
        f1 = f1_score(y_train, pred_anomaly)
        if f1 > best_f1:
            best_f1 = f1
            best_eps = eps
            best_min_samples = min_samples

print(f"Лучшие параметры: eps={best_eps}, min_samples={best_min_samples}, F1={best_f1:.3f}")

Лучшие параметры: eps=0.9526315789473684, min_samples=3, F1=0.023


In [199]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.values)
X_test_scaled = scaler.transform(X_test.values)
clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
clf.fit(X_train_scaled, y_train)
y_pred_train = clf.predict(X_train_scaled)
y_pred_test = clf.predict(X_test_scaled)

print("Train classification report:")
print(classification_report(y_train, y_pred_train))
print("Test classification report:")
print(classification_report(y_test, y_pred_test))

/home/alena/anaconda3/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning:

A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().



Train classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     24333
           1       1.00      1.00      1.00      7923

    accuracy                           1.00     32256
   macro avg       1.00      1.00      1.00     32256
weighted avg       1.00      1.00      1.00     32256

Test classification report:
              precision    recall  f1-score   support

           0       0.97      0.87      0.92      7823
           1       0.00      0.00      0.00       241

    accuracy                           0.84      8064
   macro avg       0.48      0.44      0.46      8064
weighted avg       0.94      0.84      0.89      8064



In [204]:
import numpy as np
import plotly.graph_objects as go
from sklearn.ensemble import IsolationForest
from sklearn.metrics import f1_score

# Приводим y к одномерному виду
y_train = np.asarray(y_train).ravel()
y_test = np.asarray(y_test).ravel()

# Обучающая выборка: только нормальные точки (класс 0)
X_train_norm = X_train_scaled[y_train == 0]

# Подберём contamination (ожидаемую долю аномалий) по лучшему F1 на тесте
best_f1 = 0
best_cont = 0.01
contaminations = np.linspace(0.01, 0.2, 20)

for cont in contaminations:
    iso = IsolationForest(contamination=cont, random_state=42)
    iso.fit(X_train_norm)
    pred = iso.predict(X_test_scaled)          # -1 аномалия, 1 норма
    pred_anomaly = (pred == -1).astype(int)
    f1 = f1_score(y_test, pred_anomaly)
    if f1 > best_f1:
        best_f1 = f1
        best_cont = cont

print(f"Лучший contamination: {best_cont:.3f}, F1 = {best_f1:.3f}")

# Обучаем финальную модель с лучшим параметром
iso_best = IsolationForest(contamination=best_cont, random_state=42)
iso_best.fit(X_train_norm)
pred_best = iso_best.predict(X_test_scaled)
anomaly_flags = (pred_best == -1)

# Статистика
print(f"Предсказано аномалий на тесте: {np.sum(anomaly_flags)} ({np.mean(anomaly_flags):.2%})")

# График
indices = np.arange(len(y_test))

fig = go.Figure()

# Истинные значения y_test (точки, окрашенные по классу)
fig.add_trace(go.Scatter(
    x=indices, y=y_test,
    mode='markers',
    marker=dict(color=y_test, colorscale='Viridis', showscale=False,
                size=4, opacity=0.7),
    name='Истинные метки',
    text=['норма' if v==0 else 'аномалия' for v in y_test],
    hoverinfo='text+x+y'
))

# Предсказанные аномалии (красные крестики поверх)
anom_idx = indices[anomaly_flags]
anom_vals = y_test[anomaly_flags]
fig.add_trace(go.Scatter(
    x=anom_idx, y=anom_vals,
    mode='markers',
    marker=dict(symbol='x', color='red', size=10),
    name='Предсказанные аномалии'
))

fig.update_layout(
    title='Isolation Forest: предсказанные аномалии на фоне истинных меток',
    xaxis_title='Индекс',
    yaxis_title='Класс (0 – норма, 1 – аномалия)',
    hovermode='closest'
)
fig.show()

Лучший contamination: 0.200, F1 = 0.010
Предсказано аномалий на тесте: 6318 (78.35%)


In [209]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import f1_score, make_scorer
from scipy.stats import randint, uniform

# Приводим y к одномерному виду
y_train = np.asarray(y_train).ravel()
y_test = np.asarray(y_test).ravel() if y_test is not None else None

# Разделяем обучающую выборку на train и validation (стратифицированно)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_scaled, y_train, test_size=0.2, random_state=42, stratify=y_train
)

# Для обучения Isolation Forest берём только нормальные объекты (класс 0) из train
X_tr_norm = X_tr[y_tr == 0]

# Создаём кастомный скоринг для RandomizedSearchCV
def isolation_f1(estimator, X, y_true):
    """Оценивает F1-score для IsolationForest на данных X, y_true."""
    y_pred = estimator.predict(X)
    y_pred_bin = (y_pred == -1).astype(int)  # -1 -> аномалия (1)
    return f1_score(y_true, y_pred_bin)

scorer = make_scorer(isolation_f1, greater_is_better=True)

# Определяем сетку гиперпараметров
param_dist = {
    'n_estimators': randint(50, 300),
    'max_samples': uniform(0.3, 0.7),  # от 0.3 до 1.0
    'contamination': uniform(0.01, 0.2),  # от 0.01 до 0.21
    'max_features': ['sqrt', 'log2', 0.5, 0.7, 1.0],
    'bootstrap': [True, False]
}

# Базовый оценщик
iso_base = IsolationForest(random_state=42)

# RandomizedSearchCV
random_search = RandomizedSearchCV(
    iso_base, param_distributions=param_dist,
    n_iter=50, scoring=scorer, cv=3,
    random_state=42, verbose=1, n_jobs=-1
)

# Обучаем на нормальных данных из train (X_tr_norm) и передаём валидационные данные для скоринга
# Но RandomizedSearchCV сам разбивает переданные данные на фолды.
# Важно: IsolationForest должен обучаться только на нормальных данных, поэтому в каждом фолде
# нужно брать только нормальные объекты из train_fold. Стандартный CV этого не умеет.
# Поэтому проще сделать ручной перебор с фиксированной валидационной выборкой, либо использовать
# специальную обёртку. Для упрощения реализуем ручной перебор.

# Ручной перебор (более прозрачный и контролируемый)
print("Ручной перебор гиперпараметров...")
best_f1 = 0
best_params = None
best_model = None

# Задаём сетку вручную (можно расширить)
n_estimators_list = [50, 100, 200]
max_samples_list = [0.3, 0.5, 0.7, 1.0]
contamination_list = np.linspace(0.01, 0.2, 10)
max_features_list = [0.5, 0.7, 1.0]
bootstrap_list = [True, False]

total_combinations = (len(n_estimators_list) * len(max_samples_list) *
                      len(contamination_list) * len(max_features_list) *
                      len(bootstrap_list))
print(f"Всего комбинаций: {total_combinations}")

# Для ускорения можно использовать случайные комбинации, но для полноты реализуем полный перебор
# с ограничением числа комбинаций (можно заменить на random.sample)
import itertools
from tqdm import tqdm

# Ограничим число комбинаций до 200 для скорости (можно увеличить)
param_combinations = list(itertools.product(
    n_estimators_list, max_samples_list, contamination_list,
    max_features_list, bootstrap_list
))
if len(param_combinations) > 200:
    import random
    param_combinations = random.sample(param_combinations, 200)

for n_est, max_samp, cont, max_feat, boot in tqdm(param_combinations):
    model = IsolationForest(
        n_estimators=n_est,
        max_samples=max_samp,
        contamination=cont,
        max_features=max_feat,
        bootstrap=boot,
        random_state=42
    )
    model.fit(X_tr_norm)  # обучаем только на нормальных объектах train
    y_val_pred = model.predict(X_val)
    y_val_pred_bin = (y_val_pred == -1).astype(int)
    f1 = f1_score(y_val, y_val_pred_bin)
    if f1 > best_f1:
        best_f1 = f1
        best_params = {
            'n_estimators': n_est,
            'max_samples': max_samp,
            'contamination': cont,
            'max_features': max_feat,
            'bootstrap': boot
        }
        best_model = model

print(f"\nЛучшие параметры: {best_params}")
print(f"Лучший F1 на валидации: {best_f1:.4f}")

# Обучаем финальную модель на всех нормальных объектах исходной обучающей выборки
X_train_norm_all = X_train_scaled[y_train == 0]
final_model = IsolationForest(**best_params, random_state=42)
final_model.fit(X_train_norm_all)

# Предсказание на тесте
y_test_pred = final_model.predict(X_test_scaled)
y_test_pred_bin = (y_test_pred == -1).astype(int)
test_f1 = f1_score(y_test, y_test_pred_bin)
print(f"F1 на тесте: {test_f1:.4f}")
print(f"Доля предсказанных аномалий на тесте: {np.mean(y_test_pred_bin):.2%}")

# Визуализация на тесте
indices_test = np.arange(len(y_test))
fig = go.Figure()

# Истинные метки (цветные точки)
fig.add_trace(go.Scatter(
    x=indices_test, y=y_test,
    mode='markers',
    marker=dict(color=y_test, colorscale='Viridis', showscale=False,
                size=4, opacity=0.7),
    name='Истинные метки',
    text=['норма' if v==0 else 'аномалия' for v in y_test],
    hoverinfo='text+x+y'
))

# Предсказанные аномалии (красные крестики)
anom_idx = indices_test[y_test_pred_bin == 1]
anom_vals = y_test[anom_idx]
fig.add_trace(go.Scatter(
    x=anom_idx, y=anom_vals,
    mode='markers',
    marker=dict(symbol='x', color='red', size=10),
    name='Предсказанные аномалии'
))

fig.update_layout(
    title='Isolation Forest (подобранные параметры) – предсказания на тесте',
    xaxis_title='Индекс',
    yaxis_title='Класс (0 – норма, 1 – аномалия)',
    hovermode='closest'
)
fig.show()

Ручной перебор гиперпараметров...
Всего комбинаций: 720


100%|██████████| 200/200 [01:50<00:00,  1.82it/s]



Лучшие параметры: {'n_estimators': 50, 'max_samples': 0.5, 'contamination': 0.01, 'max_features': 1.0, 'bootstrap': False}
Лучший F1 на валидации: 0.8914
F1 на тесте: 0.0047
Доля предсказанных аномалий на тесте: 76.33%


In [206]:
from sklearn.svm import OneClassSVM
svm = OneClassSVM(nu=0.03, kernel='rbf', gamma='scale')
svm.fit(X_train_norm)
pred_svm = svm.predict(X_test_scaled)   # -1 аномалия, 1 норма

fig = go.Figure()

# Истинные значения y_test (точки, окрашенные по классу)
fig.add_trace(go.Scatter(
    x=indices, y=y_test,
    mode='markers',
    marker=dict(color=y_test, colorscale='Viridis', showscale=False,
                size=4, opacity=0.7),
    name='Истинные метки',
    text=['норма' if v==0 else 'аномалия' for v in y_test],
    hoverinfo='text+x+y'
))

# Предсказанные аномалии (красные крестики поверх)
anom_idx = indices[pred_svm]
anom_vals = y_test[pred_svm]
fig.add_trace(go.Scatter(
    x=anom_idx, y=anom_vals,
    mode='markers',
    marker=dict(symbol='x', color='red', size=10),
    name='Предсказанные аномалии'
))

fig.update_layout(
    title='Isolation Forest: предсказанные аномалии на фоне истинных меток',
    xaxis_title='Индекс',
    yaxis_title='Класс (0 – норма, 1 – аномалия)',
    hovermode='closest'
)
fig.show()

In [165]:
# Индексы для оси X
indices = np.arange(len(y_test))
anomaly = [1 if i[0] == -1 else 0 for i in anomaly_flags]

# Создаём DataFrame для удобства
df_test = pd.DataFrame({
    'index': indices,
    'y_test': np.ravel(y_test),
    'anomaly': np.ravel(anomaly)  # True для аномалий
})

# Строим scatter-график, цветом выделяем аномалии
fig = px.scatter(
    df_test, x='index', y='y_test', color='anomaly',
    title='Тестовые данные с выделением аномалий',
    labels={'index': 'Индекс', 'y_test': 'Значение y_test'},
    color_discrete_map={False: 'blue', True: 'red'}
)

# Добавляем линии для лучшей читаемости
fig.update_traces(marker=dict(size=5), selector=dict(mode='markers'))
fig.add_scatter(x=indices, y=y_test, mode='lines', line=dict(color='lightgray', width=1), showlegend=False)

fig.show()

IndexError: invalid index to scalar variable.

In [ ]:
# Индексы для оси X
indices = np.arange(len(y_train))
anomaly = [1 if i[0] == -1 else 0 for i in anomaly_flags]

# Создаём DataFrame для удобства
df_test = pd.DataFrame({
    'index': indices,
    'y_test': np.ravel(y_test),
    'anomaly': np.ravel(anomaly)  # True для аномалий
})

# Строим scatter-график, цветом выделяем аномалии
fig = px.scatter(
    df_test, x='index', y='y_test', color='anomaly',
    title='Тестовые данные с выделением аномалий',
    labels={'index': 'Индекс', 'y_test': 'Значение y_test'},
    color_discrete_map={False: 'blue', True: 'red'}
)

# Добавляем линии для лучшей читаемости
fig.update_traces(marker=dict(size=5), selector=dict(mode='markers'))
fig.add_scatter(x=indices, y=y_test, mode='lines', line=dict(color='lightgray', width=1), showlegend=False)

fig.show()